1. Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split 
from tensorflow.keras import Input, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Concatenate

2. Load Data

In [2]:
# dataset path
DATA_PATH = "../../annotations"
os.listdir(DATA_PATH)

['lisa_dataset_split.csv', 'new_annotations.csv']

In [3]:
# Load the precomputed split.
# Expects 'lisa_dataset_split.csv' to be located under DATA_PATH.
df = pd.read_csv(os.path.join(DATA_PATH,'new_annotations.csv'), sep=';')

In [4]:
# Preview first 5 rows
df.head()

,image_id,label,isNight,split
0,../../datasets/processed_images\train\img_0.jpg,1,0,train
1,../../datasets/processed_images\train\img_1.jpg,1,0,train
2,../../datasets/processed_images\train\img_2.jpg,1,0,train
3,../../datasets/processed_images\train\img_3.jpg,1,0,train
4,../../datasets/processed_images\train\img_4.jpg,1,0,train


In [5]:
# Preview first 5 rows
df.shape

(51826, 4)

3. Build RNN-ready datasets

In [6]:
CSV_PATH   = "../../annotations/new_annotations.csv"
IMG_SIZE   = 50
NUM_CLASSES = 3  # stop / warning / go mapped to 0..2

# Auto-detect separator (handles ';' from previous exports)
df = pd.read_csv(CSV_PATH, sep=None, engine="python")

# Quick sanity checks on splits and label balance
print("Splits:", df["split"].value_counts().to_dict())
print("Labels per split:\n", pd.crosstab(df["split"], df["label"]))

# Labels come as 1/2/3 -> remap to 0/1/2
df["label"] = df["label"].astype(int) - 1
assert df["label"].between(0, NUM_CLASSES-1).all(), "Found labels outside 0..2"

def build_Xy(df_part):
    """
    Build design matrix and one-hot targets from a dataframe slice.

    - Reads each image path, resizes to IMG_SIZE x IMG_SIZE, normalizes to [0,1]
    - Flattens the image to a single vector
    - Appends the binary context feature `isNight` as an extra column
    - One-hot encodes labels to shape (N, NUM_CLASSES)
    """
    X_images, X_isNight, y = [], [], []

    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        img_path = str(row["image_id"]).replace("\\", "/")
        label    = int(row["label"])
        is_night = int(row["isNight"])

        # Skip missing or unreadable images
        if not os.path.exists(img_path):
            print(f"❌ Missing image: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read: {img_path}")
            continue

        # Resize and normalize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0

        # Collect features and target
        X_images.append(img)
        X_isNight.append(is_night)
        y.append(label)

    # Flatten images: (N, H, W, C) -> (N, H*W*C)
    X_images = np.array(X_images).reshape(len(X_images), -1)
    # Context feature as (N, 1)
    X_isNight = np.array(X_isNight).reshape(-1, 1)
    # Concatenate image vector with isNight -> (N, H*W*C + 1)
    X_combined = np.hstack([X_images, X_isNight])

    # One-hot encode targets
    y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
    return X_combined, y

# Build split-specific datasets from the 'split' column
X_train, y_train = build_Xy(df[df["split"] == "train"])
X_val,   y_val   = build_Xy(df[df["split"] == "val"])
X_test,  y_test  = build_Xy(df[df["split"] == "test"])

print("Shapes:",
      "\n X_train:", X_train.shape, " y_train:", y_train.shape,
      "\n X_val:  ", X_val.shape,   " y_val:  ", y_val.shape,
      "\n X_test: ", X_test.shape,  " y_test: ", y_test.shape)

Splits: {'train': 30707, 'test': 14433, 'val': 6686}
Labels per split:
 label      1    2      3
split                   
test    7229  499   6705
train  14272  800  15635
val     2681  256   3749


100%|██████████| 14433/14433 [05:22<00:00, 44.82it/s]


Shapes: 
 X_train: (30707, 7501)  y_train: (30707, 3) 
 X_val:   (6686, 7501)  y_val:   (6686, 3) 
 X_test:  (14433, 7501)  y_test:  (14433, 3)


4. Baseline CNN classifier

In [7]:
IMG_SIZE = 50
PIXELS = IMG_SIZE * IMG_SIZE * 3  # 50x50 RGB → 7,500 features

def split_back(X):
    """
    Undo the earlier feature concatenation:
    - First PIXELS columns: flattened image (HxWxC).
    - Last column: auxiliary scalar feature (isNight ∈ {0,1}).

    Returns
    -------
    X_img : np.ndarray, shape (N, IMG_SIZE, IMG_SIZE, 3)
        Image tensor reconstructed from the flat vector.
    X_isn : np.ndarray, shape (N, 1)
        Auxiliary feature as a column vector (isNight).
    """
    # Slice out the flattened image part and reshape to HxWxC
    X_img_flat = X[:, :PIXELS]
    X_img = X_img_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 3).astype("float32")

    # Take the last column as the isNight flag and keep it as (N,1)
    X_isn = X[:, -1].reshape(-1, 1).astype("float32")   
    return X_img, X_isn

# Apply to each split
Ximg_train, isn_train = split_back(X_train)
Ximg_val,   isn_val   = split_back(X_val)
Ximg_test,  isn_test  = split_back(X_test)

# Sanity check: expect (N, 50, 50, 3) for images and (N, 1) for isNight
print("Ximg_train:", Ximg_train.shape, "isn_train:", isn_train.shape) 

Ximg_train: (30707, 50, 50, 3) isn_train: (30707, 1)


Reshape images into row-wise

In [8]:
def image_to_row_sequence(Ximg):
    """
    Convert images from (N, H, W, C) to (N, H, W*C), i.e., each image becomes a
    sequence of H rows, where each timestep is the concatenation of the row's pixels
    across all channels.

    Example with 50x50x3:
      (N, 50, 50, 3) -> (N, 50, 150), where 150 = 50 * 3.
    This prepares data for sequence models (RNN/LSTM) that process one row per timestep.
    """
    N, H, W, C = Ximg.shape
    return Ximg.reshape(N, H, W*C).astype("float32")

# Build row-wise sequences for train/val/test splits
Xseq_train = image_to_row_sequence(Ximg_train)
Xseq_val   = image_to_row_sequence(Ximg_val)
Xseq_test  = image_to_row_sequence(Ximg_test)

print("Xseq_train:", Xseq_train.shape)  # Expected: (N, 50, 150) for 50x50x3 inputs


Xseq_train: (30707, 50, 150)


Build a bidirectional LSTM classifier (row-sequence + isNight)

In [9]:
# Number of target classes inferred from one-hot labels
NUM_CLASSES = y_train.shape[1]

# Inputs:
# - seq_in: row-wise sequence per image, shape (H, W*C) -> here (50, 150)
# - isn_in: scalar context feature indicating night/day (0/1)
seq_in = Input(shape=(IMG_SIZE, IMG_SIZE*3), name="row_sequence")  # (50,150)
isn_in = Input(shape=(1,), name="isNight")

# Sequence encoder:
# Bidirectional LSTM over rows; final hidden state encodes the whole image
z = Bidirectional(LSTM(64, return_sequences=False))(seq_in)
z = Dropout(0.3)(z) # regularization

# Fuse sequence features with context (isNight)
h = Concatenate()([z, isn_in])
h = Dense(128, activation='relu')(h)

# Softmax head
out = Dense(NUM_CLASSES, activation='softmax')(h)

# Compile model
rnn_model = Model([seq_in, isn_in], out)
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
rnn_model.summary()




Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 row_sequence (InputLayer)   [(None, 50, 150)]            0         []                            
                                                                                                  
 bidirectional (Bidirection  (None, 128)                  110080    ['row_sequence[0][0]']        
 al)                                                                                              
                                                                                                  
 dropout (Dropout)           (None, 128)                  0         ['bidirectional[0][0]']       
                                                                                                  
 isNight (InputLayer)        [(None, 1)]                  0         []                      

5. Training loop with early stopping

In [10]:
# Stop training when validation loss hasn’t improved for 5 epochs,
# and roll back to the best weights observed during training.
early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# ----- Fit -----
history_rnn = rnn_model.fit(
    [Xseq_train, isn_train], y_train,            # multi-input: images + isNight
    validation_data=([Xseq_val, isn_val], y_val),
    epochs=50, 
    batch_size=32, 
    shuffle=True,                                 # shuffle each epoch
    callbacks=[early], 
    verbose=1
)


Epoch 1/50


960/960 [==============================] - 20s 18ms/step - loss: 0.0630 - accuracy: 0.9795 - val_loss: 0.0285 - val_accuracy: 0.9918
Epoch 2/50
960/960 [==============================] - 16s 17ms/step - loss: 0.0220 - accuracy: 0.9928 - val_loss: 0.0326 - val_accuracy: 0.9888
Epoch 3/50
960/960 [==============================] - 17s 17ms/step - loss: 0.0176 - accuracy: 0.9940 - val_loss: 0.0441 - val_accuracy: 0.9850
Epoch 4/50
960/960 [==============================] - 16s 17ms/step - loss: 0.0132 - accuracy: 0.9959 - val_loss: 0.0750 - val_accuracy: 0.9711
Epoch 5/50
960/960 [==============================] - 17s 17ms/step - loss: 0.0124 - accuracy: 0.9962 - val_loss: 0.0185 - val_accuracy: 0.9978
Epoch 6/50
960/960 [==============================] - 17s 17ms/step - loss: 0.0122 - accuracy: 0.9959 - val_loss: 0.0190 - val_accuracy: 0.9952
Epoch 7/50
960/960 [==============================] - 17s 18ms/step - loss: 0.0123 - accuracy: 0.9962 - val_loss: 0.0126 - val_accurac

6. Evaluation

In [11]:
# Evaluate on held-out test set
test_loss_rnn, test_acc_rnn = rnn_model.evaluate([Xseq_test, isn_test], y_test, verbose=0)
print(f"RNN → Test loss: {test_loss_rnn:.4f} | Test acc: {test_acc_rnn:.4f}")

RNN → Test loss: 0.0093 | Test acc: 0.9977


In [12]:
# Predict class probabilities on the test set (batched for speed/memory).
y_pred = rnn_model.predict([Xseq_test, isn_test], batch_size=32, verbose=0)

# Convert one-hot (or probas) to hard class indices.
y_pred_idx = y_pred.argmax(axis=1)
y_true_idx = y_test.argmax(axis=1)

# Per-class precision/recall/F1 + macro/micro averages.
print(classification_report(y_true_idx, y_pred_idx, digits=4))

# Raw confusion matrix counts: rows = true, columns = predicted.
print(confusion_matrix(y_true_idx, y_pred_idx))

              precision    recall  f1-score   support

           0     0.9999    0.9960    0.9979      7229
           1     0.9861    0.9940    0.9900       499
           2     0.9963    0.9999    0.9981      6705

    accuracy                         0.9977     14433
   macro avg     0.9941    0.9966    0.9953     14433
weighted avg     0.9977    0.9977    0.9977     14433

[[7200    6   23]
 [   1  496    2]
 [   0    1 6704]]


7. Save the trained model

In [13]:
import os

MODEL_DIR = "../../models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "traffic_light_rnn.keras")

rnn_model.save(MODEL_PATH)